# ⚡ Pipeline de Vetorização Espectral SWeeP 600D na GPU (Dataset Fujita)

Este notebook implementa a versão **ultrarrápida acelerada por hardware (NVIDIA GPU)** do pipeline de vetorização celular Fujita com **operações nativas em Float32**, substituindo o pipeline em CPU/R (`vetorizar.ipynb`) e alimentando diretamente o classificador MLP (`classificar.ipynb`):
1. **Detecção e Inicialização de GPU:** Utilização do pacote **`sweep` v2.2.0+ (Python)** com backend **CuPy** em precisão **Float32 nativa** (otimizada para Tensor Cores da NVIDIA GeForce RTX 3060);
2. **Carregamento OOM-Safe:** Matriz transcriptômica Fujita (`matrizFiltradaeNormalizadaF (1).h5ad`) e rótulos (`cell_types_binarioF.txt`);
3. **Filtragem das Classes Canônicas:** Classes 1 a 7, ajustadas para 0 a 6 (zero-indexed Keras);
4. **Binarização In-place OOM-Safe:** Expressão gênica ($>0 \to 1.0$) no vetor esparso `.data` diretamente em `float32` sem densificação;
5. **Ortobase Canônica SWeeP (Float32 Chunked):** Resolução analítica da base ortonormal ($36.601 \times 600$) via `sweep.orth_base` com geração ultrarrápida particionada em blocos e formato `float32` (~84 MB em disco);
6. **Projeção Espectral Ultrarrápida na GPU:** Execução direta com `sweep.sweep(..., device='cuda')` em apenas ~2 segundos (ganho de ~58x sobre os ~115s da CPU);
7. **Exportação dos Artefatos Canônicos Prontos:** Persistência direta em `dados/X_fujita_600d.npy` (`float32`), `dados/y_fujita.npy` (`int64`), `dados/orthbase_fujita_600d.npy` (`float32`) e `dados/metadata_classes.json`, 100% compatíveis com o notebook `classificar.ipynb`.


## ⚙️ Configuração do Ambiente e Telemetria de Hardware (GPU)


In [ ]:
import json
import time
from pathlib import Path

import anndata as ad
import cupy as cp
import numpy as np
import scipy.sparse as sp
import sweep

# Parâmetros Fundamentais do Experimento
SEED = 42
DIM_PROJECAO = 600
CLASSES_VALIDAS = list(range(1, 8))  # Classes canônicas 1 a 7

# Caminhos de Entrada
PATH_DADOS = Path("dados")
#PATH_H5AD = PATH_DADOS / "fujita" / "matrizFiltradaeNormalizadaF (1).h5ad"
#PATH_LABELS = PATH_DADOS / "fujita" / "cell_types_binarioF.txt"
PATH_H5AD = PATH_DADOS / "mathys" / "matriz_anotada_finalM.h5ad"
PATH_LABELS = PATH_DADOS / "mathys" / "tipos_celulares_numericoMs.txt"

# Caminhos Canônicos de Saída (Consumidos diretamente pelo classificar.ipynb)
#PATH_X_OUT = PATH_DADOS / "fujita" / "X_fujita_600d.npy"
#PATH_Y_OUT = PATH_DADOS / "fujita" / "y_fujita.npy"
#PATH_ORTHBASE_OUT = PATH_DADOS / "fujita" / "orthbase_fujita_600d.npy"
#PATH_METADATA_OUT = PATH_DADOS / "fujita" / "metadata_classes_fujita.json"
PATH_X_OUT = PATH_DADOS / "mathys" / "X_mathys_600d.npy"
PATH_Y_OUT = PATH_DADOS / "mathys" / "y_mathys.npy"
PATH_ORTHBASE_OUT = PATH_DADOS / "mathys" / "orthbase_mathys_600d.npy"
PATH_METADATA_OUT = PATH_DADOS / "mathys" / "metadata_classes_mathys.json"

# Telemetria e Verificação de Dispositivo GPU
gpu_ativa = sweep.is_gpu_available()
nome_gpu = sweep.get_gpu_device_name()
device_id = cp.cuda.Device()
mem_info = device_id.mem_info
vram_livre_gb = mem_info[0] / (1024 ** 3)
vram_total_gb = mem_info[1] / (1024 ** 3)

print("=" * 70)
print("     SWeeP GPU PIPELINE - INICIALIZAÇÃO DE HARDWARE (FLOAT32)")
print("=" * 70)
print(f"✓ Aceleração GPU detectada : {gpu_ativa}")
print(f"  • Modelo da GPU          : {nome_gpu}")
print(f"  • VRAM Total             : {vram_total_gb:.2f} GB")
print(f"  • VRAM Disponível        : {vram_livre_gb:.2f} GB")
print(f"  • Versão do SWeeP        : {getattr(sweep, 'DEFAULT_VERSION', 'v2.2.0')}")
print("  • Backend Numérico       : CuPy (CUDA Sparse/Dense FP32 Nativo)")
print("-" * 70)
print(f"  • Dataset H5AD           : {PATH_H5AD}")
print(f"  • Arquivo de Rótulos     : {PATH_LABELS}")
print(f"  • Dimensão Projetada     : {DIM_PROJECAO}")
print(f"  • Semente Determinística : {SEED}")
print(f"  • Destino Features X     : {PATH_X_OUT}")
print(f"  • Destino Rótulos y      : {PATH_Y_OUT}")
print(f"  • Destino Ortobase       : {PATH_ORTHBASE_OUT}")
print(f"  • Destino Metadados      : {PATH_METADATA_OUT}")
print("=" * 70)
assert gpu_ativa, "ERRO: Nenhuma GPU compatível com CuPy foi detectada!"


     SWeeP GPU PIPELINE - INICIALIZAÇÃO DE HARDWARE (FLOAT32)
✓ Aceleração GPU detectada : True
  • Modelo da GPU          : NVIDIA GeForce RTX 3060
  • VRAM Total             : 11.63 GB
  • VRAM Disponível        : 9.13 GB
  • Versão do SWeeP        : SWeeP v2.9
  • Backend Numérico       : CuPy (CUDA Sparse/Dense FP32 Nativo)
----------------------------------------------------------------------
  • Dataset H5AD           : dados/matrizFiltradaeNormalizadaF (1).h5ad
  • Arquivo de Rótulos     : dados/cell_types_binarioF.txt
  • Dimensão Projetada     : 600
  • Semente Determinística : 42
  • Destino Features X     : dados/X_fujita_600d.npy
  • Destino Rótulos y      : dados/y_fujita.npy
  • Destino Ortobase       : dados/orthbase_fujita_600d.npy
  • Destino Metadados      : dados/metadata_classes.json


## 🏷️ Etapa 1: Carregamento dos Rótulos e Filtragem das Classes 1 a 7

Lemos o arquivo `cell_types_binarioF.txt` e construímos a máscara booleana para reter exclusivamente as classes celulares 1 a 7, descartando a classe 0 (não-anotadas) e classes acima de 7. Reindexamos as classes para o intervalo **0 a 6** (formato canônico zero-indexed de funções de perda como `sparse_categorical_crossentropy` do Keras).


In [2]:
t_inicio_etapa = time.time()

with open(PATH_LABELS, "r") as f:
    rotulos_originais = np.array(
        [int(l.strip()) for l in f if l.strip()], dtype=np.int64
    )

total_celulas_bruto = len(rotulos_originais)
print(f"Total de células anotadas no arquivo: {total_celulas_bruto:,}")

# Máscara de seleção para classes canônicas 1 a 7
mascara_classes = np.isin(rotulos_originais, CLASSES_VALIDAS)
total_filtradas = int(mascara_classes.sum())

# Rótulos ajustados para zero-index (0 a 6)
y_filtrado = rotulos_originais[mascara_classes] - 1

print(f"✓ Filtragem concluída em {time.time() - t_inicio_etapa:.2f}s")
print(
    f"  • Células mantidas (classes 1-7) : {total_filtradas:,} ({total_filtradas / total_celulas_bruto * 100:.2f}%)"
)
print(
    f"  • Células descartadas (classe 0/+) : {total_celulas_bruto - total_filtradas:,}"
)
print("  • Distribuição por classe ajustada (0 a 6):")
for cls_idx in range(7):
    qtd = int((y_filtrado == cls_idx).sum())
    print(f"    - Classe Keras {cls_idx} (Original {cls_idx + 1}): {qtd:,} células")


Total de células anotadas no arquivo: 40,913
✓ Filtragem concluída em 0.01s
  • Células mantidas (classes 1-7) : 26,763 (65.41%)
  • Células descartadas (classe 0/+) : 14,150
  • Distribuição por classe ajustada (0 a 6):
    - Classe Keras 0 (Original 1): 4,366 células
    - Classe Keras 1 (Original 2): 188 células
    - Classe Keras 2 (Original 3): 9,073 células
    - Classe Keras 3 (Original 4): 4,567 células
    - Classe Keras 4 (Original 5): 1,734 células
    - Classe Keras 5 (Original 6): 5,762 células
    - Classe Keras 6 (Original 7): 1,073 células


## 📦 Etapa 2: Carregamento do Dataset Transcriptômico Fujita (`.h5ad`)

Carregamos o dataset AnnData e aplicamos a máscara booleana celular para carregar apenas as 26.763 células selecionadas na memória de trabalho.


In [3]:
t_inicio_etapa = time.time()
print(f"Carregando matriz AnnData: {PATH_H5AD}...")

adata = ad.read_h5ad(PATH_H5AD)
print(f"✓ Dataset carregado em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Dimensões originais: {adata.shape[0]:,} células × {adata.shape[1]:,} genes")
print(f"  • Tipo de representação: {type(adata.X)}")

# Aplicação da máscara de classes diretamente na matriz esparsa
X_filtrado = adata.X[mascara_classes]
nomes_genes = list(adata.var_names)
barcodes_filtrados = list(adata.obs_names[mascara_classes])

print(
    f"  • Dimensões filtradas: {X_filtrado.shape[0]:,} células × {X_filtrado.shape[1]:,} genes"
)

# Liberamos o objeto AnnData completo para otimização máxima de RAM
del adata


Carregando matriz AnnData: dados/matrizFiltradaeNormalizadaF (1).h5ad...


✓ Dataset carregado em 3.56s!
  • Dimensões originais: 40,913 células × 36,601 genes
  • Tipo de representação: <class 'scipy.sparse._csc.csc_matrix'>


  • Dimensões filtradas: 26,763 células × 36,601 genes


## ⚡ Etapa 3: Binarização In-Place OOM-Safe da Expressão Gênica (Float32)

Conforme requisito metodológico da pesquisa, aplicamos a binarização:
- Expressão $> 0 \implies 1.0$
- Expressão $= 0 \implies 0.0$

> **Otimização Float32:** A matriz esparsa é estruturada diretamente em `float32`, consumindo metade da memória de arrays 64-bit e alinhando perfeitamente com os Tensor Cores da GPU NVIDIA.


In [4]:
t_inicio_etapa = time.time()

# Asseguramos formato CSR em float32 para máxima velocidade em hardware CUDA
if sp.issparse(X_filtrado):
    X_bin = X_filtrado.tocsr()
    X_bin.data = np.ones_like(X_bin.data, dtype=np.float32)
else:
    X_bin = sp.csr_matrix((X_filtrado > 0).astype(np.float32))

print(f"✓ Binarização concluída em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Total de entradas ativas (1s): {X_bin.nnz:,}")
print(
    f"  • Densidade da matriz esparsa  : {(X_bin.nnz / (X_bin.shape[0] * X_bin.shape[1])) * 100:.2f}%"
)
print(f"  • Tipo de dado do vetor .data  : {X_bin.data.dtype}")
print(f"  • Valores únicos em .data      : {np.unique(X_bin.data)}")


✓ Binarização concluída em 2.15s!
  • Total de entradas ativas (1s): 104,708,300
  • Densidade da matriz esparsa  : 10.69%
  • Tipo de dado do vetor .data  : float32


  • Valores únicos em .data      : [1.]


## 📐 Etapa 4: Resolução e Preparação da Ortobase Canônica SWeeP (Float32 Nativo)

O algoritmo **SWeeP** (*Spaced Words Projection*) mapeia os $36.601$ genes para um subespaço latente de **600 dimensões** através de uma base ortonormal $B \in \mathbb{R}^{36.601 \times 600}$:
$$B^T B \approx I_{600}$$

No **`sweep` v2.2.0+ (Python)** atualizado, a função `sweep.orth_base` gera a base diretamente em **`float32`** com algoritmo particionado em blocos (*chunked in-place*), gerando a base completa em menos de **0.2 segundos** e ocupando apenas **83.77 MB** em disco.


In [5]:
t_inicio_etapa = time.time()
n_genes = X_bin.shape[1]

print(f"Dimensões do espaço de features: {n_genes:,} genes")
print(f"Dimensão alvo da projeção SWeeP: {DIM_PROJECAO} coordenadas")
print(f"Semente determinística         : {SEED}")

if PATH_ORTHBASE_OUT.exists():
    print(f"\n✓ Carregando e reutilizando ortobase existente em: {PATH_ORTHBASE_OUT}")
    orthbase_mat = np.load(PATH_ORTHBASE_OUT).astype(np.float32)
    assert orthbase_mat.shape == (n_genes, DIM_PROJECAO), (
        f"Incompatibilidade dimensional: esperava ({n_genes}, {DIM_PROJECAO}), encontrou {orthbase_mat.shape}"
    )
    print(f"  • Shape confirmado: {orthbase_mat.shape}")
    print(f"  • Tipo de dado    : {orthbase_mat.dtype}")
else:
    print("\n• Gerando nova ortobase canônica analítica via sweep.orth_base (Float32 nativo)...")
    t_gen = time.time()
    orth_res = sweep.orth_base(lin=n_genes, col=DIM_PROJECAO, seed=SEED, dtype=np.float32)
    orthbase_mat = orth_res.mat
    print(f"  ✓ Ortobase gerada em {time.time() - t_gen:.2f}s! Shape: {orthbase_mat.shape}, Dtype: {orthbase_mat.dtype}")

# Garante persistência em float32 no disco (83.77 MB)
if not PATH_ORTHBASE_OUT.exists() or PATH_ORTHBASE_OUT.stat().st_size > 100 * 1024 * 1024:
    np.save(PATH_ORTHBASE_OUT, orthbase_mat)
    print(f"  ✓ Ortobase gravada em float32 com sucesso em: {PATH_ORTHBASE_OUT}")

print(f"✓ Ortobase pronta para projeção em {time.time() - t_inicio_etapa:.2f}s!")


Dimensões do espaço de features: 36,601 genes
Dimensão alvo da projeção SWeeP: 600 coordenadas
Semente determinística         : 42

✓ Carregando e reutilizando ortobase existente em: dados/orthbase_fujita_600d.npy
  • Shape confirmado: (36601, 600)
  • Tipo de dado    : float32
  ✓ Ortobase gravada em float32 com sucesso em: dados/orthbase_fujita_600d.npy
✓ Ortobase pronta para projeção em 0.17s!


## 🚀 Etapa 5: Projeção Espectral Ultrarrápida na GPU (CuPy em Float32)

Invocamos a função homologada **`sweep.sweep()`** com `device="cuda"`. A matriz esparsa binarizada $X_{\text{bin}} \in \mathbb{R}^{26.763 \times 36.601}$ e a base $B \in \mathbb{R}^{36.601 \times 600}$ são processadas em núcleos CUDA nativos pelo **CuPy** em **FP32**.

Com a atualização para `float32`, a multiplicação matricial esparsa atinge velocidade recorde de **~2 segundos** (aceleração de **~58x** em relação aos ~115s do R em CPU).


In [6]:
t_inicio_etapa = time.time()

print("=" * 70)
print("     INICIANDO PROJEÇÃO ESPECTRAL SWEEP NA GPU (CUDA FP32)")
print("=" * 70)
print(f"• Dispositivo de Execução  : {sweep.get_gpu_device_name()}")
print(f"• Matriz de Entrada        : {X_bin.shape[0]:,} células × {X_bin.shape[1]:,} genes")
print(f"• Dimensão Alvo            : {DIM_PROJECAO}")
print(f"• Precisão Numérica        : Float32 (Tensor Cores / Alta Performance)")

# Telemetria de VRAM antes da projeção
mempool = cp.get_default_memory_pool()
mempool.free_all_blocks()
vram_usada_inicio = mempool.used_bytes() / (1024 ** 2)

# Execução da projeção espectral nativa em GPU (FP32)
t0_proj = time.time()
resultado_sweep = sweep.sweep(
    input=X_bin,
    orthbase=orthbase_mat,
    device="cuda",
    dtype=np.float32,
    verbose=True,
)
tempo_proj = time.time() - t0_proj

X_projetado = resultado_sweep.proj
vram_usada_pico = mempool.used_bytes() / (1024 ** 2)
throughput = X_bin.shape[0] / tempo_proj

print("\n✓ PROJEÇÃO CONCLUÍDA COM SUCESSO!")
print(f"  • Tempo total de projeção GPU : {tempo_proj:.2f} segundos")
print(f"  • Throughput celular          : {throughput:.1f} células/segundo")
print(f"  • Aceleração vs R CPU (~115s) : {115 / tempo_proj:.1f}x mais rápido!")
print(f"  • VRAM CuPy alocada em pico   : {vram_usada_pico:.1f} MB")
print(f"  • Dimensão da matriz projetada: {X_projetado.shape}")
print(f"  • Tipo de dado resultante     : {X_projetado.dtype}")
print(f"  • Quantidade de NaNs          : {np.isnan(X_projetado).sum()}")
print(f"  • Quantidade de Infs          : {np.isinf(X_projetado).sum()}")
print(f"  • Média geral dos valores     : {X_projetado.mean():.4f}")
print(f"  • Desvio padrão geral         : {X_projetado.std():.4f}")
print("=" * 70)


     INICIANDO PROJEÇÃO ESPECTRAL SWEEP NA GPU (CUDA FP32)
• Dispositivo de Execução  : NVIDIA GeForce RTX 3060
• Matriz de Entrada        : 26,763 células × 36,601 genes
• Dimensão Alvo            : 600
• Precisão Numérica        : Float32 (Tensor Cores / Alta Performance)



✓ PROJEÇÃO CONCLUÍDA COM SUCESSO!
  • Tempo total de projeção GPU : 1.97 segundos
  • Throughput celular          : 13561.3 células/segundo
  • Aceleração vs R CPU (~115s) : 58.3x mais rápido!
  • VRAM CuPy alocada em pico   : 0.0 MB
  • Dimensão da matriz projetada: (26763, 600)
  • Tipo de dado resultante     : float32
  • Quantidade de NaNs          : 0
  • Quantidade de Infs          : 0
  • Média geral dos valores     : -0.0036
  • Desvio padrão geral         : 0.3259


## 💾 Etapa 6: Persistência Canônica dos Dados de Entrada para a Rede MLP

Salvamos os tensores canônicos em formato binário NumPy (`.npy`) e o arquivo de metadados (`.json`). A matriz de features `X_fujita_600d.npy` ocupa **61.26 MB**, formato nativo consumido pelo Keras/TensorFlow para inferência e treinamento sem necessidade de conversão.


In [7]:
t_inicio_etapa = time.time()

np.save(PATH_X_OUT, X_projetado)
np.save(PATH_Y_OUT, y_filtrado)
np.save(PATH_ORTHBASE_OUT, orthbase_mat)

# Metadados completos do experimento com rastreabilidade científica total
metadados = {
    "nome_dataset": "Fujita Filtrado e Normalizado (Binarizado) - GPU SWeeP FP32",
    "algoritmo": "SWeeP (Spaced Words Projection - Implementação Oficial Python / CuPy GPU)",
    "pacote_versao": getattr(sweep, "DEFAULT_VERSION", "v2.2.0"),
    "dispositivo_execucao": str(sweep.get_gpu_device_name()),
    "backend": "CuPy CUDA Sparse-Dense Matmul (FP32)",
    "dimensao_latente": DIM_PROJECAO,
    "semente_ortobase": SEED,
    "total_amostras_originais": total_celulas_bruto,
    "total_amostras_mlp": int(X_projetado.shape[0]),
    "total_genes": int(X_bin.shape[1]),
    "tempo_projecao_segundos": round(tempo_proj, 2),
    "throughput_celulas_por_seg": round(throughput, 1),
    "dtype_final": str(X_projetado.dtype),
    "mapeamento_classes": {
        f"Classe_Keras_{i}": {
            "label_original": i + 1,
            "label_keras": i,
            "total_amostras": int((y_filtrado == i).sum()),
        }
        for i in range(7)
    },
    "arquivos_gerados": {
        "features_x": str(PATH_X_OUT),
        "labels_y": str(PATH_Y_OUT),
        "ortobase": str(PATH_ORTHBASE_OUT),
    },
}

with open(PATH_METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(metadados, f, indent=2, ensure_ascii=False)

tam_x_mb = PATH_X_OUT.stat().st_size / (1024 * 1024)
tam_y_kb = PATH_Y_OUT.stat().st_size / 1024
tam_base_mb = PATH_ORTHBASE_OUT.stat().st_size / (1024 * 1024)

print(f"✓ Artefatos canônicos persistidos em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Features X     : {PATH_X_OUT} ({tam_x_mb:.2f} MB, dtype: {X_projetado.dtype})")
print(f"  • Labels y       : {PATH_Y_OUT} ({tam_y_kb:.2f} KB, shape: {y_filtrado.shape})")
print(f"  • Ortobase       : {PATH_ORTHBASE_OUT} ({tam_base_mb:.2f} MB, dtype: {orthbase_mat.dtype})")
print(f"  • Metadados      : {PATH_METADATA_OUT}")


✓ Artefatos canônicos persistidos em 0.03s!
  • Features X     : dados/X_fujita_600d.npy (61.26 MB, dtype: float32)
  • Labels y       : dados/y_fujita.npy (209.21 KB, shape: (26763,))
  • Ortobase       : dados/orthbase_fujita_600d.npy (83.77 MB, dtype: float32)
  • Metadados      : dados/metadata_classes.json


## 🔍 Etapa 7: Verificação e Auditoria de Prontidão para o Keras (classificar.ipynb)

Demonstramos a carga instantânea dos arquivos gerados e confirmamos todos os tipos de dados e limites esperados pela rede MLP para classificação celular.


In [8]:
# Carga instantânea via NumPy para auditoria estrita
X_verif = np.load(PATH_X_OUT)
y_verif = np.load(PATH_Y_OUT)
orth_verif = np.load(PATH_ORTHBASE_OUT)

print("=" * 60)
print("     AUDITORIA DE PRONTIDÃO PARA A REDE MLP KERAS")
print("=" * 60)
print(f"• X shape        : {X_verif.shape} (esperado: (26763, 600))")
print(f"• X dtype        : {X_verif.dtype} (esperado: float32)")
print(f"• y shape        : {y_verif.shape} (esperado: (26763,))")
print(f"• y dtype        : {y_verif.dtype} (esperado: int64)")
print(f"• y min / max    : {y_verif.min()} / {y_verif.max()} (esperado: 0 a 6)")
print(f"• Classes únicas : {np.unique(y_verif)}")
print(f"• Ortobase shape : {orth_verif.shape} (esperado: (36601, 600))")
print(f"• Ortobase dtype : {orth_verif.dtype} (esperado: float32)")
print(f"• Ausência de NaN: {not np.isnan(X_verif).any()}")
print(f"• Ausência de Inf: {not np.isinf(X_verif).any()}")
print("=" * 60)
print("🎯 DADOS 100% PRONTOS PARA EXECUÇÃO NO CLASSIFICAR.IPYNB!")


     AUDITORIA DE PRONTIDÃO PARA A REDE MLP KERAS
• X shape        : (26763, 600) (esperado: (26763, 600))
• X dtype        : float32 (esperado: float32)
• y shape        : (26763,) (esperado: (26763,))
• y dtype        : int64 (esperado: int64)
• y min / max    : 0 / 6 (esperado: 0 a 6)
• Classes únicas : [0 1 2 3 4 5 6]
• Ortobase shape : (36601, 600) (esperado: (36601, 600))
• Ortobase dtype : float32 (esperado: float32)
• Ausência de NaN: True
• Ausência de Inf: True
🎯 DADOS 100% PRONTOS PARA EXECUÇÃO NO CLASSIFICAR.IPYNB!
